In [1]:
from diffforms import *
from sympy import sin, cos, sqrt #Sympy functions for things

# Abreviate long commands
PD = PartialDerivative
CD = CovariantDerivative
CT = Contract
PI = PermuteIndices

In [10]:
M = Manifold("M", 4, signature=[-1,1,1,1]) 

coords = t,r,theta,phi = scalars(r"t r \theta \phi",real=True) # Coordinates are sympy Symbols
mass = constants("M") # constants define symbols that are not treated as coordinates
Cosmo = constants("Λ")

M.set_coordinates(coords) # Give manifold coordinates to produce basis for vectors and differential forms

basis = dt,dr,dtheta,dphi = M.get_basis()   # Basis elements for the cotangent space 
vects = vt,vr,vtheta,vphi = M.get_vectors() # Basis elements for the tangent space

# Define Metric components using sympy "Function"s
F = Function(r"f",real=True,positive=True)(r)
G = Function(r"g",real=True,positive=True)(r)

e_I = e0,e1,e2,e3 = F*dt, G*dr, r*dtheta, r*sin(theta)*dphi # Define frame 1-forms

M.set_frame(e_I) # Give frame to Manifold for frame/metric dependent computations

g_DD = M.get_metric() # Computes metric from frame
g_UU = M.get_inverse_metric() # and its inverse

G_UDD = M.get_christoffel_symbols() # Computes the Christoffel symbols for the intrinsic metric on the manifold

print("Metric:")
display(g_DD) # Displays the metric
print("Christoffel Symbols:")
G_UDD # Displays the Christoffel symbols

Metric:


$(- f^{2})d\left(t\right) \otimes d\left(t\right)+(g^{2})d\left(r\right) \otimes d\left(r\right)+(r^{2})d\left(\theta\right) \otimes d\left(\theta\right)+(r^{2} \sin^{2}{\left(\theta \right)})d\left(\phi\right) \otimes d\left(\phi\right)$

Christoffel Symbols:


$(\frac{\frac{d}{d r} f}{f})\partial_{t} \otimes d\left(r\right) \otimes d\left(t\right)+(\frac{\frac{d}{d r} g}{g})\partial_{r} \otimes d\left(r\right) \otimes d\left(r\right)+(\frac{1}{r})\partial_{\theta} \otimes d\left(r\right) \otimes d\left(\theta\right)+(\frac{1}{r})\partial_{\phi} \otimes d\left(r\right) \otimes d\left(\phi\right)+(\frac{1}{\tan{\left(\theta \right)}})\partial_{\phi} \otimes d\left(\theta\right) \otimes d\left(\phi\right)+(\frac{\frac{d}{d r} f}{f})\partial_{t} \otimes d\left(t\right) \otimes d\left(r\right)+(\frac{1}{r})\partial_{\theta} \otimes d\left(\theta\right) \otimes d\left(r\right)+(\frac{1}{r})\partial_{\phi} \otimes d\left(\phi\right) \otimes d\left(r\right)+(\frac{1}{\tan{\left(\theta \right)}})\partial_{\phi} \otimes d\left(\phi\right) \otimes d\left(\theta\right)+(\frac{f \frac{d}{d r} f}{g^{2}})\partial_{r} \otimes d\left(t\right) \otimes d\left(t\right)+(- \frac{r}{g^{2}})\partial_{r} \otimes d\left(\theta\right) \otimes d\left(\theta\right)+(- 

### Solutions to Einstein's Equations

In [11]:
G_DD = M.get_einstein_tensor().simplify()

EE_DD = G_DD + Cosmo*g_DD

EE_DD

$(- f^{2} \text{Λ} + \frac{f^{2} \left(g^{3} - g + 2 r \frac{d}{d r} g\right)}{g^{3} r^{2}})d\left(t\right) \otimes d\left(t\right)+(r^{2} \text{Λ} + \frac{r \left(- f \frac{d}{d r} g + g r \frac{d^{2}}{d r^{2}} f + g \frac{d}{d r} f - r \frac{d}{d r} f \frac{d}{d r} g\right)}{f g^{3}})d\left(\theta\right) \otimes d\left(\theta\right)+(r^{2} \sin^{2}{\left(\theta \right)} \text{Λ} + \frac{r \left(- f \frac{d}{d r} g + g r \frac{d^{2}}{d r^{2}} f + g \frac{d}{d r} f - r \frac{d}{d r} f \frac{d}{d r} g\right) \sin^{2}{\left(\theta \right)}}{f g^{3}})d\left(\phi\right) \otimes d\left(\phi\right)+(g^{2} \text{Λ} - \frac{g^{2}}{r^{2}} + \frac{1}{r^{2}} + \frac{2 \frac{d}{d r} f}{f r})d\left(r\right) \otimes d\left(r\right)$

In [12]:
from sympy import dsolve

Gsoleq = dsolve(CT(EE_DD*vt*vt,(0,2),(1,3)).simplify()*G**3*r**2/F**2,G)[1]
Gsol = {Gsoleq.lhs: Gsoleq.rhs}


Fsoleq = dsolve(CT(EE_DD*vr*vr,(0,2),(1,3)).subs(Gsol).simplify(),F)
FGsol = {Gsoleq.lhs: Gsoleq.rhs, Fsoleq.lhs: Fsoleq.rhs}


g_DD.subs(FGsol).simplify().simplify()

$(\frac{C_{2}^{2} \left(- r^{3} \text{Λ} + C_{1} + 3 r\right)}{r})d\left(t\right) \otimes d\left(t\right)+(\frac{3 r}{- r^{3} \text{Λ} + C_{1} + 3 r})d\left(r\right) \otimes d\left(r\right)+(r^{2})d\left(\theta\right) \otimes d\left(\theta\right)+(r^{2} \sin^{2}{\left(\theta \right)})d\left(\phi\right) \otimes d\left(\phi\right)$

### SU(2) Structure Objects

In [13]:
from diffforms.gstructures.SU2 import *
S_i = S1,S2,S3 = GetSU2Structures(e_I)

A_i = A1,A2,A3 = [a.simplify() for a in GetSU2Connections(S_i)]

F_i = F1,F2,F3 = [f.simplify() for f in GetSU2Curvature(A_i)]

print("Self-dual 2-forms:")
for s in S_i:
    display(s)

print("self-dual connections:")
for a in A_i:
    display(a)

print("self-dual curvature")
for f in F_i:
    display(f)

Self-dual 2-forms:


self-dual connections:


self-dual curvature


In [ ]:
R_DD = GetSU2MetricIrreducibleFromTwoFormTriple(F_i,S_i).simplify()
g_DD = GetUrbantkeMetric(S_i)
g_UU = Rank2TensorInverse(g_DD)

R    = Contract(R_DD*g_UU,(0,2),(1,3)).simplify()

EE_DD = (R_DD - Number(1,2)*R*g_DD + Cosmo*g_DD).simplify()

$(g^{2} \text{Λ} + \frac{g^{2}}{r^{2}} - \frac{1}{r^{2}} - \frac{2 \frac{d}{d r} f}{f r})d\left(r\right) \otimes d\left(r\right)+(\frac{f^{2} \left(- g^{3} r^{2} \text{Λ} - g^{3} + g - 2 r \frac{d}{d r} g\right)}{g^{3} r^{2}})d\left(t\right) \otimes d\left(t\right)+(\frac{r \left(f g^{3} r \text{Λ} + f \frac{d}{d r} g - g r \frac{d^{2}}{d r^{2}} f - g \frac{d}{d r} f + r \frac{d}{d r} f \frac{d}{d r} g\right) \sin^{2}{\left(\theta \right)}}{f g^{3}})d\left(\phi\right) \otimes d\left(\phi\right)+(\frac{r \left(f g^{3} r \text{Λ} + f \frac{d}{d r} g - g r \frac{d^{2}}{d r^{2}} f - g \frac{d}{d r} f + r \frac{d}{d r} f \frac{d}{d r} g\right)}{f g^{3}})d\left(\theta\right) \otimes d\left(\theta\right)$

$(- \frac{2 \frac{d}{d r} g}{g r} + \frac{g \frac{d^{2}}{d r^{2}} f - \frac{d}{d r} f \frac{d}{d r} g}{f g})d\left(r\right) \otimes d\left(r\right)+(- \frac{2 f \frac{d}{d r} f}{g^{2} r} - \frac{f \left(g \frac{d^{2}}{d r^{2}} f - \frac{d}{d r} f \frac{d}{d r} g\right)}{g^{3}})d\left(t\right) \otimes d\left(t\right)+(\left(- \sin{\left(\theta \right)} + \frac{\sin{\left(\theta \right)}}{g^{2}}\right) \sin{\left(\theta \right)} - \frac{r \sin^{2}{\left(\theta \right)} \frac{d}{d r} g}{g^{3}} + \frac{r \sin^{2}{\left(\theta \right)} \frac{d}{d r} f}{f g^{2}})d\left(\phi\right) \otimes d\left(\phi\right)+(- \frac{\sin{\left(\theta \right)} - \frac{\sin{\left(\theta \right)}}{g^{2}}}{\sin{\left(\theta \right)}} - \frac{r \frac{d}{d r} g}{g^{3}} + \frac{r \frac{d}{d r} f}{f g^{2}})d\left(\theta\right) \otimes d\left(\theta\right)$